In [13]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [14]:
data = pd.read_csv('customer_churn.csv')

In [15]:
print(data.head())
print(data.info())

   CustomerID  Tenure  MonthlyCharges  TotalCharges        Contract  \
0        1001       5            70.0         350.0  Month-to-month   
1        1002      10            85.5         850.5        Two year   
2        1003       3            55.3         165.9        One year   
3        1004       8            90.0         720.0  Month-to-month   
4        1005       2            65.2         130.4        One year   

      PaymentMethod  Churn  
0  Electronic check      1  
1      Mailed check      0  
2  Electronic check      1  
3       Credit card      0  
4  Electronic check      1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      5 non-null      int64  
 1   Tenure          5 non-null      int64  
 2   MonthlyCharges  5 non-null      float64
 3   TotalCharges    5 non-null      float64
 4   Contract        5 non-nu

In [16]:
# handle outliers and missing values
data = data.drop(columns=['CustomerID']) #Simplify the dataset
data = data.dropna()  # Simple example of dropping missing values

In [17]:
# Encode categorical values
data = pd.get_dummies(data, drop_first=True)

In [18]:
# Split the data for training
X = data.drop('Churn', axis=1)
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 10.3735 - val_accuracy: 1.0000 - val_loss: 9.0269e-25
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.5000 - loss: 12.1873 - val_accuracy: 1.0000 - val_loss: 1.9732e-20
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.5000 - loss: 12.4034 - val_accuracy: 1.0000 - val_loss: 6.3501e-16
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.2500 - loss: 26.7618 - val_accuracy: 1.0000 - val_loss: 1.6437e-13
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.7500 - loss: 18.6508 - val_accuracy: 1.0000 - val_loss: 4.6977e-13
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 1.0000 - loss: 0.0389 - val_accuracy: 1.0000 - val_loss: 1.0152e-12
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.7500 - loss: 4.9515 - val_accuracy: 1.0000 - val_loss: 7.0184e-12
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.5000 - loss: 20.5309 - val_accu

In [21]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f'Test accuracy: {test_acc}')

1/1 - 0s - 37ms/step - accuracy: 1.0000 - loss: 6.1888e-11
Test accuracy: 1.0


In [22]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

INFO:tensorflow:Assets written to: /tmp/tmpdqs27kp3/assets


INFO:tensorflow:Assets written to: /tmp/tmpdqs27kp3/assets


Saved artifact at '/tmp/tmpdqs27kp3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7), dtype=tf.float32, name='keras_tensor_5')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  125486091811328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125486074148320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125486074139520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125486074150080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125486074146736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125486074143392: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1758741574.182470    5225 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1758741574.182505    5225 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2025-09-24 19:19:34.182743: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpdqs27kp3
2025-09-24 19:19:34.183484: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-09-24 19:19:34.183496: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpdqs27kp3
2025-09-24 19:19:34.187718: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-09-24 19:19:34.213788: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpdqs27kp3
2025-09-24 19:19:34.222167: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 39429 microseconds.


In [23]:
model.save('churn_model.h5')